## EXPLANATION

1. **Inspect before cleaning.** The four diagnostic commands are `df.info()`, `df.dtypes`, `df.isna().sum()` (nulls per column), and `df.duplicated().sum()` (duplicate rows).
2. **Nulls in Pandas are `NaN` (or `NaT` for dates).** `isna()` and `notna()` play the role of SQL's `IS NULL` and `IS NOT NULL`, returning `True`/`False` Boolean masks that you can use for filtering.
  - **Two ways to handle nulls:** `dropna()` removes rows. With `subset=["col"]`, it checks only that specific column, and with `how="all"`, it only drops a row if all its values are null.
  - `fillna(value)` fills them in. It can be a fixed value, the mean (`df["col"].mean()`), the median, or a dictionary with a specific value per column: `df.fillna({"city": "not provided"})`. It's the equivalent of SQL's `COALESCE`.
3. **Imputation is not neutral.** Filling with `0`, the mean, or the median alters the analysis results. The median is safer than the mean when outliers are present. Decide on a column-by-column basis and document your reasoning.
4. **Duplicates:** `duplicated()` flags repeated rows after the first occurrence as `True`. `drop_duplicates()` removes those rows. With `subset=["order"]`, the comparison is based solely on that column, and with `keep="last"`, you keep the last occurrence.
5. **Type casting (conversion):** `astype(float)` converts types, but throws an error if there is non-numeric text.
  - `pd.to_numeric(col, errors="coerce")` converts values and replaces anything that fails with `NaN`.
  - `pd.to_datetime(col, format="%d/%m/%Y", errors="coerce")` converts dates, turning invalid ones into `NaT`. The `format` parameter avoids ambiguity between day/month and month/day.
  - `int64` does not support `NaN`, so an integer column with nulls becomes `float64`, unless you use the nullable `"Int64"` dtype (with a capital `I`).
6. **Text preview (the full topic is covered in Module 7):** `.str.strip()` removes leading and trailing whitespace, while `.str.lower()` and `.str.title()` standardize letter casing. Without this, `"Ana"`, `"ana "`, and `"ANA"` count as three different customers.
7. **Recommended workflow order:** First standardize text and cast data types, then remove duplicates, and finally handle missing values. Standardizing can uncover hidden duplicates, and type casting can introduce new nulls.
8. **Immutability by default:** These methods return a new DataFrame and do not modify the original one in place. Therefore, you need to reassign it: `df = df.drop_duplicates()`. Save a backup with `df_original = df.copy()` before starting.

## EXAMPLE

In [1]:
import pandas as pd

grades = pd.DataFrame({
    "class": ["A", "A", "B", "B", "B", "C"],
    "subject": ["math", "port", "math", "port", "math", "port"],
    "grade": [8.0, 6.5, 5.0, 7.5, 9.0, 4.0],
})

In [3]:
# an aggregation

grades.groupby("class")["grade"].mean().round(2)

,grade
class,
A,7.25
B,7.17
C,4.00


In [10]:
# multiple aggregations with custom names (named aggregation)

summary = grades.groupby("class").agg(
    mean=("grade", "mean"),
    max_grade=("grade", "max"),
    exams=("grade", "count"),
).round({"mean": 2})

print(summary)

       mean  max_grade  exams
class                        
A      7.25        8.0      2
B      7.17        9.0      3
C      4.00        4.0      1


In [12]:
# two keys + reset_index to turn into a regular table

by_class_subject = (
    grades.groupby(["class", "subject"])["grade"].mean().reset_index()
)

print(by_class_subject)

  class subject  grade
0     A    math    8.0
1     A    port    6.5
2     B    math    7.0
3     B    port    7.5
4     C    port    4.0


In [16]:
# frequency of each value

grades["class"].value_counts()
grades["class"].value_counts(normalize=True).round(2)   # proportions


,proportion
class,
B,0.50
A,0.33
C,0.17


In [18]:
# top 2 classes by mean

grades.groupby("class")["grade"].mean().sort_values(ascending=False).head(2).round(2)

,grade
class,
A,7.25
B,7.17


In [19]:
# pivot table

grades.pivot_table(index="class", columns="subject", values="grade",
                   aggfunc="mean", fill_value=0)

subject,math,port
class,,
A,8.0,6.5
B,7.0,7.5
C,0.0,4.0


In [21]:
# HAVING: only classes with mean above 6

means = grades.groupby("class")["grade"].mean().round(2)
means[means > 6]

,grade
class,
A,7.25
B,7.17


## EXERCISE

In [30]:
import pandas as pd
pd.set_option("display.precision", 2)

data = {
    "order": [301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312],
    "customer": ["Ana", "Bruno", "Carla", "Ana", "Diego", "Bruno",
                 "Elisa", "Carla", "Diego", "Ana", "Elisa", "Bruno"],
    "city": ["Rio", "Niterói", "Rio", "Rio", "São Gonçalo", "Niterói",
             "Rio", "Rio", "São Gonçalo", "Rio", "Rio", "Niterói"],
    "product": ["t-shirt", "sneakers", "cap", "pants", "t-shirt", "cap",
                "sneakers", "t-shirt", "pants", "sneakers", "cap", "pants"],
    "category": ["clothing", "footwear", "accessory", "clothing", "clothing", "accessory",
                 "footwear", "clothing", "clothing", "footwear", "accessory", "clothing"],
    "qty": [2, 1, 3, 1, 5, 2, 1, 4, 2, 1, 2, 3],
    "unit_price": [49.9, 249.9, 35.0, 119.9, 49.9, 35.0,
                   249.9, 49.9, 119.9, 249.9, 35.0, 119.9],
}

df = pd.DataFrame(data)
df["revenue"] = df["qty"] * df["unit_price"]

In [31]:
# Total revenue by city (revenue summed by city).

df.groupby("city")["revenue"].sum()

,revenue
city,
Niterói,679.6
Rio,1094.1
São Gonçalo,489.3


In [32]:
# Number of orders per category, in two ways: using value_counts and using
# groupby + size. The results should be the same.

print(df["category"].value_counts())
print("\n-----------------------\n")
count = df.groupby("category").size().reset_index(name="quantidade")
print(count)


category
clothing     6
footwear     3
accessory    3
Name: count, dtype: int64

-----------------------

    category  quantidade
0  accessory           3
1   clothing           6
2   footwear           3


In [35]:
# Summary per product with named aggregation: total_qty (sum of qty),
# avg_revenue (mean of revenue), and max_revenue (max revenue).

summary = df.groupby(["product"], as_index=False).agg(
    total_qty = ("qty", "sum"),
    avg_revenue = ("revenue", "mean"),
    max_revenue = ("revenue", "max")
)

print(summary)

    product  total_qty  avg_revenue  max_revenue
0       cap          7        81.67        105.0
1     pants          6       239.80        359.7
2  sneakers          3       249.90        249.9
3   t-shirt         11       182.97        249.5


In [47]:
# Revenue by city and category, using both keys, with the result as a regular
# table (with reset_index) and sorted from highest to lowest revenue.

by_city_category = (
    df.groupby(["city", "category"])["revenue"].sum().reset_index()
    .sort_values(by="revenue", ascending=False)
)
print(by_city_category)

          city   category  revenue
5          Rio   footwear    499.8
6  São Gonçalo   clothing    489.3
4          Rio   clothing    419.3
1      Niterói   clothing    359.7
2      Niterói   footwear    249.9
3          Rio  accessory    175.0
0      Niterói  accessory     70.0


In [38]:
# Pivot table with city in the rows, category in the columns, and the sum of
# revenue in the values. Combinations that do not exist should appear as 0.

df.pivot_table(index="city", columns="category", values="revenue",
               aggfunc="sum", fill_value=0)

category,accessory,clothing,footwear
city,,,
Niterói,70.0,359.7,249.9
Rio,175.0,419.3,499.8
São Gonçalo,0.0,489.3,0.0


In [43]:
# The top 3 customers by spending, showing the name and total spent,
# from highest to lowest.

top3 = (
    df.groupby("customer", as_index=False)["revenue"].sum()
    .sort_values(by="revenue", ascending=False)
    .head(3)
)

print(top3)

  customer  revenue
1    Bruno    679.6
3    Diego    489.3
0      Ana    469.6


In [49]:
# BONUS: Calculate the percentage of total revenue for each city
# (for example, Rio = 62.5%). Think about how to divide each city's sum by
# the total sum, and round to 1 decimal place.

summary_city = df.groupby("city", as_index=False)["revenue"].sum()

summary_city["percentage"] = (summary_city["revenue"] / summary_city["revenue"]
                              .sum() * 100).round(1)

summary_city = summary_city.sort_values(by="percentage", ascending=False)
print(summary_city)

          city  revenue  percentage
1          Rio   1094.1        48.3
0      Niterói    679.6        30.0
2  São Gonçalo    489.3        21.6
